In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# 加载训练好的模型
model_path = "my_model_1"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("模型加载成功！")

def predict_sentiment(text):
    """
    预测单条文本的情绪
    返回: 情绪标签和置信度
    """
    # 编码文本
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=128
    )
    
    # 模型预测
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # 获取预测结果
    confidence, predicted_class = torch.max(predictions, dim=1)
    confidence = confidence.item()
    predicted_class = predicted_class.item()
    
    # 映射到情绪标签 (根据你的数据: 0=正面, 1=负面)
    sentiment = "正面" if predicted_class == 0 else "负面"
    
    return {
        'text': text,
        'label': predicted_class,
        'confidence': round(confidence, 4),
        'sentiment': sentiment
    }



OSError: my_model_1 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [4]:
# 测试一些例子
test_texts = [
    "这个产品非常好用，质量很棒！",
    "服务态度太差了，很不满意",
    "物流速度很快，包装完好",
    "价格太贵，性价比不高",
    "效果很不错，会再次购买",
    "商品有瑕疵，质量有问题",
    "草你妈",
    "你好帅啊"
]

print("=== 情绪分析测试 ===")
for text in test_texts:
    result = predict_sentiment(text)
    print(f"文本: {result['text']}")
    print(f"情绪: {result['sentiment']} (置信度: {result['confidence']:.2%})")
    print("-" * 60)

=== 情绪分析测试 ===
文本: 这个产品非常好用，质量很棒！
情绪: 正面 (置信度: 66.94%)
------------------------------------------------------------
文本: 服务态度太差了，很不满意
情绪: 负面 (置信度: 77.38%)
------------------------------------------------------------
文本: 物流速度很快，包装完好
情绪: 正面 (置信度: 62.40%)
------------------------------------------------------------
文本: 价格太贵，性价比不高
情绪: 负面 (置信度: 74.84%)
------------------------------------------------------------
文本: 效果很不错，会再次购买
情绪: 正面 (置信度: 67.73%)
------------------------------------------------------------
文本: 商品有瑕疵，质量有问题
情绪: 负面 (置信度: 82.33%)
------------------------------------------------------------
文本: 草你妈
情绪: 负面 (置信度: 83.91%)
------------------------------------------------------------
文本: 你好帅啊
情绪: 负面 (置信度: 73.56%)
------------------------------------------------------------
